# 1. Load dataloaders 

In [ ]:
from data.dataloader import create_dataloaders

train_dataloader, val_dataloader = create_dataloaders(
    healthy_dir= "../project_datasets/drawing/Healthy/",
    pd_dir= "../project_datasets/drawing/Parkinson/",
    
    img_size=(256, 256),
    batch_size= 32,
)

# 2. Load model

In [ ]:
from Models.mobilenetV3 import MobileNetV3LargeBinary

model = MobileNetV3LargeBinary()
model_name = "Spiral_Drawing_Model"

# 3. Train models

In [ ]:
from training.trainer import train


train(
    model= model,
    train_dataloader=  train_dataloader,
    val_dataloader=  val_dataloader,
    
    model_name= model_name,
    run_name= model_name,
    
    load_pretrained="checkpoints/MobileNetV3-Large_best.pth",
    
    epochs= 50
)

In [ ]:
# !tensorboard --logdir=runs

# 4. Plot confusion matrix (of the best model)

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# load model
from Models.mobilenetV3 import MobileNetV3LargeBinary

model = MobileNetV3LargeBinary()
checkpoint = torch.load("checkpoints/Spiral_Drawing_Model_best.pth", map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

print(f"Loaded pretrained model:")
print(f"- val_loss={checkpoint['val_loss']:.4f}")
print(f"- val_acc={checkpoint['val_acc']:.4f}")
print(f"- val_recall={checkpoint['val_recall']:.4f}")
print(f"- val_precision={checkpoint['val_precision']:.4f}")
print(f"- val_f1={checkpoint['val_f1']:.4f}")

In [ ]:
from training.confusion_mat import plot_confusion_matrix

plot_confusion_matrix(
    model=model,
    dataloader=val_dataloader,
    device=device,
    class_names=["Healthy", "PD"],
    # threshold=0.49,
)

In [ ]:
from twilio.rest import Client
from dotenv import load_dotenv
import os

# Load environment variables from .env
load_dotenv()

# Example: access a variable
account_sid = os.getenv("SID")
auth_token = os.getenv("AUTH_CODE")
from_number = os.getenv("FROM_NUMBER")
to_number = os.getenv("TO_NUMBER")
content = os.getenv("CONTENT")

client = Client(account_sid, auth_token)

message = client.messages.create(
    from_=from_number,
    content_sid=content,
    content_variables='{"1":"12/1","2":"3pm"}',
    to=to_number
)